# NB-02 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-02**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-02.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

Use this after NB-01 to distinguish marginal losses, transmission-limit penalties, and the separate cash-flow terms in settlement.

- [Volume I, §7.1 — A declared loss model and its marginal prices](https://sherandoresearch.github.io/power-markets-series/vol1_congestion_lmp.html#s7-1)
- [Volume I, §8 — Shadow-price caps](https://sherandoresearch.github.io/power-markets-series/vol1_congestion_lmp.html#s8)
- [Volume I, §9 — Settlement, fully assembled](https://sherandoresearch.github.io/power-markets-series/vol1_congestion_lmp.html#s9)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-02', require_key=True)
print(setup_status)


# NB-02 — Losses, penalties, and the settlement ledger
Companion to Volume I §§7–10. Prerequisite: NB-01's dispatch and price signs.
We reproduce the corrected three-bus reference-allocated quadratic-loss fixture,
reconcile its cash ledger, and contrast it with the book's separate lossless
soft-limit example. All model inputs already exist in the manuscript/fixtures.
No random samples, market-year generation, or API acquisition occurs.

1. Verify the saved loss cases and energy balance.
2. Decompose prices and reconcile settlement cash flows.
3. Verify the load-bus marginal price numerically.
4. Compare congestion rights with total-price spreads and soft-limit penalties.

In [ ]:
NOTEBOOK_ID='NB-02'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return {k:response[k] for k in ('result','diagnostics')}
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

cases=copy.deepcopy(inputs['source_metadata']);derivatives={};settlements={}
print('Locally inspectable original model inputs:',inputs['requests']['uncongested-300.0']['request']['inputs'])
for name in ('uncongested','congested'):
    label=name+'-300.0';entry=inputs['requests'][label]
    output=execute(label,entry['method'],entry['request'])
    row=output['result']['dispatch_result'];cases[name]=row
    derivatives[name]=output['diagnostics']['load_price_derivative_check']['central_cost_derivative']
    q=dict(inputs=dict(declared_loss_model=inputs['settlement']['declared_loss_model'],delivery_contract=inputs['settlement']['delivery_contract'],dispatch_result={k:row[k] for k in inputs['settlement']['dispatch_fields']}),parameters={},context=dict(input_origin='Actual private NB02 optimizer output projected to existing settlement diagnostic fields; original one-hour mathematical example'))
    checked=execute(name+':settlement','loss_settlement',q,[label])['result']['settlement_ledger']
    settlements[name]=checked
    for key in ('loss_mw','load_payment','generation_payment','congestion_rent','loss_surplus','settlement_residual'):
        np.testing.assert_allclose(checked[key],row[key],atol=1e-8,rtol=0)
print({k:cases[k] for k in ('model','loss_coefficient_per_mw','loss_withdrawal_allocation','branch_order')})
for name in ('uncongested','congested'):
    row=cases[name];g=np.asarray(row['generation_mw'])
    np.testing.assert_allclose(g.sum(),row['load_mw']+row['loss_mw'],atol=1e-9)
    print(name,{'generation_MW':g.round(6).tolist(),'load_MW':row['load_mw'],'loss_MW':row['loss_mw'],'flows_MW':row['transfer_flows_mw']})

## What this approximation means
Transfer flows use lossless DC sensitivities. Quadratic withdrawals
r Σ f², with r=0.0002 MW⁻¹, are allocated entirely to bus 3. Thus generation
exceeds customer load, but this is not a branch-end AC model. The extra
withdrawals close the declared nodal balance; the transfer flows themselves
are not an AC measurement of sending- and receiving-end power.

The branch order is 1→3, 2→3, 1→2, unlike NB-01's order. Bus 3 is the reference.
Price = energy + congestion + loss component under this allocation convention.
Negative loss components at generator buses do not mean negative physical losses.

In [ ]:
plt=setup_plot();fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,name in zip(axes,('uncongested','congested')):
    row=cases[name];energy=np.full(3,row['energy_price']);cong=np.asarray(row['congestion_components']);loss=np.asarray(row['loss_components'])
    np.testing.assert_allclose(energy+cong+loss,row['nodal_prices'],atol=1e-10)
    x=np.arange(3)
    for offset,values,label,color in [(-.24,energy,'Energy','#88C0D0'),(0,cong,'Congestion','#B48EAD'),(.24,loss,'Loss','#D08770')]:
        ax.bar(x+offset,values,.24,label=label,color=color)
    ax.plot(x,row['nodal_prices'],'o',color='#ECEFF4',label='Total price')
    ax.set_xticks(x,['Bus 1','Bus 2','Bus 3']);ax.set(title=name,ylabel='Toy currency/MWh');ax.axhline(0,color='#4C566A')
axes[1].legend(loc='upper left',bbox_to_anchor=(1,1));fig.tight_layout();plt.show()

## The one-hour cash ledger
Customer load payment minus generation payment equals congestion rent plus
loss surplus in this declared model. Generation payment is not generation cost;
offers and nodal prices coincide only for interior marginal generators.
Loss surplus is not another physical loss quantity. Do not mix MW, MWh, and cash.

In [ ]:
ledger=[]
for name in ('uncongested','congested'):
    r=cases[name];net=r['load_payment']-r['generation_payment']
    np.testing.assert_allclose(net,r['congestion_rent']+r['loss_surplus'],atol=1e-8)
    ledger.append({'case':name,**{k:r[k] for k in ('load_payment','generation_payment','congestion_rent','loss_surplus','settlement_residual')}})
    print(json.dumps(ledger[-1],indent=2))
fig,ax=plt.subplots(figsize=(10,4));x=np.arange(2)
ax.bar(x-.18,[r['load_payment']-r['generation_payment'] for r in ledger],.36,color='#88C0D0',label='Load payment − generator payment')
ax.bar(x+.18,[r['congestion_rent'] for r in ledger],.36,color='#A3BE8C',label='Congestion rent')
ax.bar(x+.18,[r['loss_surplus'] for r in ledger],.36,bottom=[r['congestion_rent'] for r in ledger],color='#D08770',label='Loss surplus')
ax.set_xticks(x,['Uncongested','Congested']);ax.set(ylabel='Toy currency per one-hour case',title='Settlement reconciliation')
ax.legend();fig.tight_layout();plt.show()

## Marginal-price and instrument checks
Central differences of optimized generation cost with respect to bus-3 load
reproduce its price. The numerical step (10⁻³ MW) is the existing correction
test's step, not a new load dataset. We stay inside each analytic active regime.
A nominal 1 MW source-1/sink-3 congestion obligation earns the congestion spread
for one hour, not the total LMP spread. Funding, acquisition, fees and tariff
eligibility are outside this gross-payoff comparison.

In [ ]:
checks=[]
for name in ('uncongested','congested'):
    r=cases[name];fd=derivatives[name]
    assert abs(fd-r['nodal_prices'][2])<1e-5
    cong=r['congestion_components'][2]-r['congestion_components'][0]
    loss=r['loss_components'][2]-r['loss_components'][0]
    total=r['nodal_prices'][2]-r['nodal_prices'][0]
    np.testing.assert_allclose(cong+loss,total,atol=1e-10)
    checks.append({'case':name,'finite_difference_price':fd,'reported_price':r['nodal_prices'][2],
        'one_MW_one_hour_congestion_cash':cong,'loss_spread_cash':loss,'total_spread_cash':total})
    print(checks[-1])
# Tolerances above are numerical fixture checks. PJM data is not used here.
# NB-13 applies its own descriptor-specific DA component tolerance; rounded RT
# observations must not inherit that much tighter DA tolerance.

## A different meaning of “penalty”: soft transmission limits
Reuse Volume I §6.3's lossless wind example: offers −25 and 30, load 300 MW,
wind capability 400 MW, and line 1→3 limit 150 MW. With a hard limit, μ=165.
Section 8's illustrative P=100 makes violation cheaper than redispatch; its
P=4,500 example provides a high-cap comparison. These are toy penalties,
not current tariff values. No lossy-case price is mixed into this example.

Both directional violations have nonnegative slack and a linear penalty.
Stationarity gives μ≤P; a positive slack pins its dual to P. The objective
includes the penalty, so compare production cost and penalty cost separately.

In [ ]:
penalties=[]
for label in ('soft-None','soft-100.0','soft-4500.0'):
    entry=inputs['requests'][label]
    row=execute(label,entry['method'],entry['request'])['result']['dispatch_result']
    penalties.append({k:np.asarray(v) if isinstance(v,list) else v for k,v in row.items()})

for r in penalties:
    np.testing.assert_allclose(r['objective'],r['generation_cost']+r['penalty_cost'],atol=1e-7)
    if r['penalty'] is not None:
        np.testing.assert_allclose(r['directional_slack_mw']*(r['penalty']-r['directional_duals']),0,atol=1e-7)
    print({k:(v.tolist() if isinstance(v,np.ndarray) else v) for k,v in r.items()})
fig,axes=plt.subplots(1,2,figsize=(12,4));labels=['Hard limit','P = 100','P = 4,500']
axes[0].bar(range(3),[r['flow_mw'][1] for r in penalties],color='#88C0D0');axes[0].axhline(150,color='#D08770',ls='--',label='Declared 150 MW limit');axes[0].legend(loc='upper left')
axes[0].set_xticks(range(3),labels);axes[0].set(title='Soft-limit response',ylabel='Line 1→3 flow, MW')
axes[1].bar(range(3),[r['directional_duals'][0] for r in penalties],color='#B48EAD');axes[1].set_xticks(range(3),labels)
axes[1].set(title='Shadow price',ylabel='Toy currency/MWh');fig.tight_layout();plt.show()

## Exercise
In the uncongested lossy case, can a total-price spread be positive while the
congestion obligation pays zero? Use the component ledger above. Then explain
why the soft-limit penalty is neither a marginal loss factor nor an FTR fee.
Answer: loss components can differ without congestion; the penalty prices a
violation in an explicitly different optimization problem.

In [ ]:
for control in inputs['negative_controls']:
    reject(control['method'],control['request'])
checked=dict(losses=cases,ledger=ledger,price_checks=checks,penalties=penalties,settlements=settlements,new_synthetic_data=False,scientific_acceptance=False)
(session.output/'consumer-summary.json').write_text(json.dumps(checked,default=lambda x:x.tolist(),indent=2,allow_nan=False)+'\n')
summary=session.finish(dict(saved_loss_parity=True,energy_balance=True,settlement_balance=True,price_derivative=True,congestion_only_payoff=True,soft_limit_dual_cap=True,new_synthetic_data=False,scientific_acceptance=False))
print(summary)